In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets torch scikit-learn matplotlib pandas

In [ ]:
from pathlib import Path
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def find_dataset_path() -> Path:
    drive_root = Path('/content/drive/MyDrive')
    candidates = list(drive_root.rglob('final_dataset.csv'))
    if not candidates:
        raise FileNotFoundError('final_dataset.csv was not found under /content/drive/MyDrive')
    return candidates[0]

def detect_columns(frame: pd.DataFrame) -> tuple[str, str]:
    code_candidates = ['code', 'snippet', 'text', 'source_code', 'content']
    label_candidates = ['label', 'type', 'authorship', 'class', 'target']
    code_col = next((column for column in code_candidates if column in frame.columns), None)
    label_col = next((column for column in label_candidates if column in frame.columns), None)
    if code_col is None or label_col is None:
        raise ValueError(f'Could not infer code and label columns from: {list(frame.columns)}')
    return code_col, label_col

def normalize_label(value) -> int:
    text = str(value).strip().lower()
    human_values = {'0', 'human', 'human-written', 'human_written', 'written', 'manual'}
    ai_values = {'1', 'ai', 'ai-generated', 'ai_generated', 'generated', 'machine', 'machine-generated', 'machine_generated'}
    if text in human_values:
        return 0
    if text in ai_values:
        return 1
    try:
        return int(float(text))
    except ValueError as error:
        raise ValueError(f'Unsupported label value: {value!r}') from error

DATA_PATH = find_dataset_path()
frame = pd.read_csv(DATA_PATH)
code_column, label_column = detect_columns(frame)
frame = frame[[code_column, label_column]].dropna().rename(columns={code_column: 'code', label_column: 'label'})
frame['code'] = frame['code'].astype(str)
frame['label'] = frame['label'].apply(normalize_label).astype(int)
train_df, test_df = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=frame['label'])
test_df = test_df.reset_index(drop=True)
test_df.head()

In [ ]:
TOKEN_PATTERN = re.compile(r'[A-Za-z_]\w*|\d+\.\d+|\d+|==|!=|<=|>=|->|::|<<|>>|//|/\*|\*/|[^\s]')
MAX_LENGTH = 512

def tokenize_code(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(str(text))

def build_textcnn_model(checkpoint_path: Path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    vocabulary = checkpoint['vocab']

    class TextCNN(torch.nn.Module):
        def __init__(self, vocab_size: int, embed_dim: int = 128, num_filters: int = 100):
            super().__init__()
            self.embedding = torch.nn.Embedding(vocab_size, embed_dim, padding_idx=0)
            self.convs = torch.nn.ModuleList([torch.nn.Conv1d(embed_dim, num_filters, kernel_size) for kernel_size in (3, 4, 5)])
            self.dropout = torch.nn.Dropout(0.5)
            self.fc = torch.nn.Linear(num_filters * 3, 1)
            self.sigmoid = torch.nn.Sigmoid()

        def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
            embedded = self.embedding(input_ids).transpose(1, 2)
            features = [torch.relu(conv(embedded)) for conv in self.convs]
            pooled = [torch.max(feature, dim=2).values for feature in features]
            concatenated = torch.cat(pooled, dim=1)
            logits = self.fc(self.dropout(concatenated)).squeeze(1)
            return self.sigmoid(logits)

    model = TextCNN(len(vocabulary))
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return model, vocabulary

def encode_textcnn_batch(code_batch, vocabulary):
    encoded = []
    for code in code_batch:
        token_ids = [vocabulary.get(token, vocabulary['<unk>']) for token in tokenize_code(code)][:MAX_LENGTH]
        if len(token_ids) < MAX_LENGTH:
            token_ids.extend([vocabulary['<pad>']] * (MAX_LENGTH - len(token_ids)))
        encoded.append(token_ids)
    return torch.tensor(encoded, dtype=torch.long)

textcnn_checkpoint = Path('/content/drive/MyDrive/models/textcnn_weights.pt')
if not textcnn_checkpoint.exists():
    raise FileNotFoundError('TextCNN checkpoint not found at /content/drive/MyDrive/models/textcnn_weights.pt')

textcnn_model, textcnn_vocab = build_textcnn_model(textcnn_checkpoint)
textcnn_model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

codebert_path = Path('/content/drive/MyDrive/models/codebert_finetuned')
if not codebert_path.exists():
    raise FileNotFoundError('CodeBERT model folder not found at /content/drive/MyDrive/models/codebert_finetuned')

codebert_tokenizer = AutoTokenizer.from_pretrained(str(codebert_path))
codebert_model = AutoModelForSequenceClassification.from_pretrained(str(codebert_path))
codebert_model.eval()
codebert_model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
textcnn_model = textcnn_model.to(device)
codebert_model = codebert_model.to(device)

def evaluate_textcnn(data_frame: pd.DataFrame):
    predictions = []
    targets = []
    start_time = time.perf_counter()
    batch_size = 16

    with torch.no_grad():
        for start_index in range(0, len(data_frame), batch_size):
            batch = data_frame.iloc[start_index:start_index + batch_size]
            inputs = encode_textcnn_batch(batch['code'].tolist(), textcnn_vocab).to(device)
            outputs = textcnn_model(inputs)
            batch_predictions = (outputs >= 0.5).long().cpu().numpy().tolist()
            predictions.extend(batch_predictions)
            targets.extend(batch['label'].tolist())

    total_time_ms = (time.perf_counter() - start_time) * 1000
    return np.array(targets), np.array(predictions), total_time_ms / max(1, len(data_frame))

def evaluate_codebert(data_frame: pd.DataFrame):
    predictions = []
    targets = []
    start_time = time.perf_counter()
    batch_size = 16

    with torch.no_grad():
        for start_index in range(0, len(data_frame), batch_size):
            batch = data_frame.iloc[start_index:start_index + batch_size]
            encoded = codebert_tokenizer(
                batch['code'].tolist(),
                truncation=True,
                padding='max_length',
                max_length=512,
                return_tensors='pt',
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}
            outputs = codebert_model(**encoded)
            batch_predictions = torch.argmax(outputs.logits, dim=-1).cpu().numpy().tolist()
            predictions.extend(batch_predictions)
            targets.extend(batch['label'].tolist())

    total_time_ms = (time.perf_counter() - start_time) * 1000
    return np.array(targets), np.array(predictions), total_time_ms / max(1, len(data_frame))

textcnn_targets, textcnn_predictions, textcnn_avg_ms = evaluate_textcnn(test_df)
codebert_targets, codebert_predictions, codebert_avg_ms = evaluate_codebert(test_df)

textcnn_targets[:5], codebert_targets[:5]

In [ ]:
def summarize_results(name: str, y_true: np.ndarray, y_pred: np.ndarray, avg_ms: float) -> dict:
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1_Score': round(f1_score(y_true, y_pred, zero_division=0), 4),
        'Avg_Inference_ms': round(avg_ms, 4),
    }

comparison_df = pd.DataFrame([
    summarize_results('TextCNN', textcnn_targets, textcnn_predictions, textcnn_avg_ms),
    summarize_results('CodeBERT', codebert_targets, codebert_predictions, codebert_avg_ms),
])

print(comparison_df.to_string(index=False))
comparison_df

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1_Score']
x_positions = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(11, 6))
plt.bar(x_positions - width / 2, comparison_df.loc[comparison_df['Model'] == 'TextCNN', metrics].iloc[0], width, label='TextCNN', color='blue')
plt.bar(x_positions + width / 2, comparison_df.loc[comparison_df['Model'] == 'CodeBERT', metrics].iloc[0], width, label='CodeBERT', color='orange')
plt.xticks(x_positions, metrics)
plt.ylim(0, 1.05)
plt.ylabel('Score')
plt.title('TextCNN vs CodeBERT on SnapScript Test Set')
plt.legend()
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
winner_row = comparison_df.sort_values('F1_Score', ascending=False).iloc[0]
loser_row = comparison_df.sort_values('F1_Score', ascending=False).iloc[1]
metric_deltas = {
    metric: round(float(winner_row[metric] - loser_row[metric]), 4) for metric in metrics
}
speed_delta = round(float(loser_row['Avg_Inference_ms'] - winner_row['Avg_Inference_ms']), 4)

conclusion = (
    f"{winner_row['Model']} won overall on F1_Score. It led by {metric_deltas['Accuracy']:.4f} in Accuracy, "
    f"{metric_deltas['Precision']:.4f} in Precision, {metric_deltas['Recall']:.4f} in Recall, and "
    f"{metric_deltas['F1_Score']:.4f} in F1_Score. Average inference time differed by {abs(speed_delta):.4f} ms per sample."
)

print(conclusion)